# TOOLS

In [1]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.tools import tool

In [34]:
@tool
def add(a: int, b: int) -> int:
    """
    Get sum of two numbers

    Args:
        a: first number
        b: second number
    
    """
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """
    Calculate the multiplication of two numbers

    Args:
        a: First number
        b: Second number
    """

    return a * b


In [35]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()
MODEL = os.getenv("MODEL")
MODEL_PROVIDER= os.getenv("MODEL_PROVIDER")

model = init_chat_model(model=MODEL, model_provider=MODEL_PROVIDER, temperature=0.2, reasoning=False)

prompt= ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', '{question}'),
])

parser = StrOutputParser()

In [36]:
model_with_tools = model.bind_tools([add, multiply])
ai = model_with_tools.invoke("calculate 2*3 by using tool")
print(ai.tool_calls)

[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'a6f576b1-6a6c-4327-ad89-90e469f998d6', 'type': 'tool_call'}]


In [37]:
tools = [add, multiply]
tool_map = {t.name: t for t in tools}

print(tools)
print(type(tools))
print(tool_map)
print(type(tool_map))

[StructuredTool(name='add', description='Get sum of two numbers\n\nArgs:\n    a: first number\n    b: second number', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x75c08c5b3740>), StructuredTool(name='multiply', description='Calculate the multiplication of two numbers\n\nArgs:\n    a: First number\n    b: Second number', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x75c08c5b3920>)]
<class 'list'>
{'add': StructuredTool(name='add', description='Get sum of two numbers\n\nArgs:\n    a: first number\n    b: second number', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x75c08c5b3740>), 'multiply': StructuredTool(name='multiply', description='Calculate the multiplication of two numbers\n\nArgs:\n    a: First number\n    b: Second number', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x75c08c5b3920>)}
<class 'dict'>


In [40]:

messages = [HumanMessage("calculate the result of 34 * 53")]
turn = 0
while True:
    turn += 1
    ai = model_with_tools.invoke(messages)
    messages.append(ai)
    if not ai.tool_calls:
        print(f"[turn {turn}] no tool call → final: {ai.content}")
        break
    for call in ai.tool_calls:
        print(f"[turn {turn}] selected: {call['name']}  args={call['args']}")
        result = tool_map[call["name"]].invoke(call["args"])
        print(f"[turn {turn}] {call['name']} returned: {result}")
        messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))


[turn 1] selected: multiply  args={'a': 34, 'b': 53}
[turn 1] multiply returned: 1802
[turn 2] no tool call → final: The result of 34 * 53 is 1802.


In [41]:
print(messages)

[HumanMessage(content='calculate the result of 34 * 53', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-08-20T23:09:02.683415233Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1157253865, 'load_duration': 167379647, 'prompt_eval_count': 391, 'prompt_eval_duration': 461718362, 'eval_count': 38, 'eval_duration': 502144536, 'logprobs': None, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--01a0216f-5cd4-72c3-afbe-37b9446f239c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 34, 'b': 53}, 'id': '4c2697d3-2125-45db-bda1-93b3a2b2fbc9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 391, 'output_tokens': 38, 'total_tokens': 429}), ToolMessage(content='1802', tool_call_id='4c2697d3-2125-45db-bda1-93b3a2b2fbc9'), AIMessage(content='The result of 34 * 53 is 1802.', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'c

In [42]:
print(add.extras)  
print(multiply.extras)

None
None


In [ ]:
# extras: optional provider-specific fields (not used by the tool's Python code).
# name / description / args go to every model. extras go only if the provider
# understands them (e.g. Anthropic cache_control, defer_loading).

@tool(extras={"defer_loading": True, "cache_control": {"type": "ephemeral"}})
def search_files(query: str) -> str:
    """Search files by a text query."""
    return f"Found 2 files matching '{query}'"

print("search_files.extras:", search_files.extras)
print("still a normal tool:", search_files.invoke({"query": "notes"}))
